In [ ]:
# =============================================================================
# Required packages
# =============================================================================
import os
import re
from io import BytesIO
import time

import requests
import pandas as pd
import geopandas as gpd
import shapely.geometry #import Point
import matplotlib.pyplot as plt
from config import config
import numpy as np
from pathlib import Path
from typing import Optional, Union, Sequence

import bcdata
import pycancensus as pc

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)


In [ ]:
print(config)

In [ ]:
# Retrieve and set passwords 

os.environ["CANCENSUS_API_KEY"] = ""

# BCDC keys for DB and CSD census data
DB_LAYER_KEY = ""  
CSD_LAYER_KEY = ""

# Pop projections: CKAN package + resource IDs from R script
POP_PROJ_PACKAGE_ID = ""
POP_PROJ_RESOURCE_ID = ""


In [ ]:
# =============================================================================
# Parameters
# =============================================================================

CURRENT_YEAR = 2025
PROJECTION_YEARS = [CURRENT_YEAR, CURRENT_YEAR + 5, CURRENT_YEAR + 10]
CANCENSUS_YEAR = "CA21"

#File locations for input/output

from pathlib import Path

BASE_DIR = Path.cwd()
INPUT_DIR = BASE_DIR / "input"
OUTPUT_DIR = BASE_DIR / "output"

#FACILITIES_CSV = INPUT_DIR / "full-service-bc-locs-wgs84.csv"
FACILITIES_CSV = INPUT_DIR / "full-service-bc-locs-wgs84_updated.csv"
SEHI_CSV=INPUT_DIR / "SEHI-csd-weighted-scores-2025-07-21_masked_ind25.xls"


RURAL_MATRIX_XLSX = INPUT_DIR / "rural_matrix_list_of_communities.xlsx"
OUTPUT_DIRECTORY = OUTPUT_DIR

# file_path = "your file path here. May or may not work with network drives. Tested locally(ish) on onedrive."
# FACILITIES_CSV = f"{file_path}/data/test/full-service-bc-locs-wgs84.csv"
# RURAL_MATRIX_XLSX = f"{file_path}/data/test/rural_matrix_list_of_communities.xlsx"
# OUTPUT_DIRECTORY = f"{file_path}/outputs/test_py_output"

# Census area
PR = "59"

# Census vectors:
# Currently not in use as they crash the api
CENSUS_VECTORS = ["v_CA21_1"]


In [ ]:
# =============================================================================
# Helper functions
# =============================================================================

#Import Helper functions 
from HelperFunctions import *


# Summary 1: Current state: population within 15km by urban, rural and remote, indegenous 

In [ ]:
# ---------------------------------------------------------------------
# Define function
# Return a summarized table of population, indegeous, remoteness from the analysis results
# Inputs: analyisis result folder, population year - current is 2030 and distance_threshold_km 
# ---------------------------------------------------------------------


def summarize_db_access_coverage_from_path(
    db_table_path: Union[str, Path],
    population_year: Union[int, str],
    distance_threshold_km: float,
    distance_col: str = "centroid_distance_m",
    urban_rural_col: str = "urban_rural",
    indigenous_categories: Optional[Sequence[str]] = None,
    sheet_name: Optional[Union[str, int]] = 0,
    round_digits: int = 2
) -> pd.DataFrame:
    """
    Summarize population access coverage within a distance threshold from a DB table file.

    Inputs:
      - db_table_path:
            Path to DB-level table.
            Supported formats:
                .csv
                .xlsx
                .xls
                .parquet

      - population_year:
            Population year, for example 2025, 2030, or 2035.
            Function expects a column named pop_2025, pop_2030, etc.

      - distance_threshold_km:
            Distance threshold in kilometres.
            The input distance column is assumed to be in metres.

      - distance_col:
            Distance column in metres. Default: centroid_distance_m.

      - urban_rural_col:
            Urban/rural category column. Default: urban_rural.
            Expected examples:
                Urban 1
                Urban 2
                Rural 1
                Rural 2
                Rural 3
                Indigenous

      - indigenous_categories:
            Values in urban_rural_col treated as Indigenous.
            If None, defaults to ["Indigenous"].

      - sheet_name:
            Excel sheet name or index. Only used for .xlsx / .xls files.

      - round_digits:
            Number of digits for rounding numeric output.

    Output:
      - One-row DataFrame with:
            total population
            population within distance threshold
            rural population coverage
            Rural 3 population coverage
            Indigenous population coverage
    """

    # ---------------------------------------------------------------------
    # Change: Read DB table from file path instead of taking DataFrame input.
    # ---------------------------------------------------------------------
    db_table_path = Path(db_table_path)

    if not db_table_path.exists():
        raise FileNotFoundError(f"DB table path does not exist: {db_table_path}")

    file_suffix = db_table_path.suffix.lower()

    if file_suffix == ".csv":
        db_table = pd.read_csv(db_table_path, low_memory=False)

    elif file_suffix in [".xlsx", ".xls"]:
        db_table = pd.read_excel(db_table_path, sheet_name=sheet_name)

    elif file_suffix == ".parquet":
        db_table = pd.read_parquet(db_table_path)

    else:
        raise ValueError(
            "Unsupported DB table file type. "
            "Supported formats are: .csv, .xlsx, .xls, .parquet"
        )

    if not isinstance(db_table, pd.DataFrame):
        raise ValueError(
            "Loaded object is not a DataFrame. "
            "For Excel files, set sheet_name to a specific sheet name or sheet index."
        )

    # ---------------------------------------------------------------------
    # Resolve population column
    # ---------------------------------------------------------------------
    population_col = f"pop_{population_year}"

    required_cols = {
        population_col,
        distance_col,
        urban_rural_col
    }

    missing_cols = required_cols - set(db_table.columns)

    if missing_cols:
        raise ValueError(
            f"DB table missing required columns: {sorted(missing_cols)}. "
            f"Available columns: {list(db_table.columns)}"
        )

    df = db_table.copy()

    # ---------------------------------------------------------------------
    # Convert numeric columns
    # ---------------------------------------------------------------------
    df[population_col] = pd.to_numeric(
        df[population_col],
        errors="coerce"
    ).fillna(0)

    df[distance_col] = pd.to_numeric(
        df[distance_col],
        errors="coerce"
    )

    if (df[population_col] < 0).any():
        raise ValueError(f"{population_col} contains negative values.")

    # ---------------------------------------------------------------------
    # Build distance mask
    # ---------------------------------------------------------------------
    distance_threshold_m = distance_threshold_km * 1000

    # Change: Null distance is not treated as within threshold.
    within_distance_threshold = (
        df[distance_col].notna()
        & (df[distance_col] <= distance_threshold_m)
    )

    # ---------------------------------------------------------------------
    # Normalize urban/rural category values
    # ---------------------------------------------------------------------
    urban_rural_label = (
        df[urban_rural_col]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )

    # Change: Rural includes Rural 1, Rural 2, and Rural 3.
    is_rural = urban_rural_label.isin([
        "rural 1",
        "rural 2",
        "rural 3"
    ])

    is_rural_3 = urban_rural_label.eq("rural 3")

    # Change: Indigenous is now a category under urban_rural_col.
    if indigenous_categories is None:
        indigenous_categories = ["Indigenous"]

    indigenous_categories_normalized = {
        str(value).strip().lower()
        for value in indigenous_categories
    }

    is_indigenous = urban_rural_label.isin(indigenous_categories_normalized)

    # ---------------------------------------------------------------------
    # Helper functions
    # ---------------------------------------------------------------------
    def _population(mask: pd.Series) -> float:
        return float(df.loc[mask, population_col].sum())

    def _pct(numerator: float, denominator: float) -> float:
        if denominator == 0 or pd.isna(denominator):
            return np.nan
        return numerator / denominator * 100

    # ---------------------------------------------------------------------
    # Population totals
    # ---------------------------------------------------------------------
    total_population = _population(
        pd.Series(True, index=df.index)
    )

    total_population_within_threshold = _population(
        within_distance_threshold
    )

    total_rural_population = _population(
        is_rural
    )

    total_rural_population_within_threshold = _population(
        is_rural & within_distance_threshold
    )

    total_rural_3_population = _population(
        is_rural_3
    )

    total_rural_3_population_within_threshold = _population(
        is_rural_3 & within_distance_threshold
    )

    total_indigenous_population = _population(
        is_indigenous
    )

    total_indigenous_population_within_threshold = _population(
        is_indigenous & within_distance_threshold
    )

    # ---------------------------------------------------------------------
    # Build output
    # ---------------------------------------------------------------------
    result = pd.DataFrame([{
        "source_file": str(db_table_path),
        "population_year": int(population_year),
        "population_col": population_col,
        "distance_threshold_km": distance_threshold_km,
        "distance_threshold_m": distance_threshold_m,

        "total_population": total_population,
        "total_population_within_distance_threshold": total_population_within_threshold,
        "pct_population_within_distance_threshold": _pct(
            total_population_within_threshold,
            total_population
        ),

        "total_rural_population": total_rural_population,
        "total_rural_population_within_distance_threshold": total_rural_population_within_threshold,
        "pct_rural_population_within_distance_threshold": _pct(
            total_rural_population_within_threshold,
            total_rural_population
        ),

        "total_rural_3_population": total_rural_3_population,
        "total_rural_3_population_within_distance_threshold": total_rural_3_population_within_threshold,
        "pct_rural_3_population_within_distance_threshold": _pct(
            total_rural_3_population_within_threshold,
            total_rural_3_population
        ),

        "total_indigenous_population": total_indigenous_population,
        "total_indigenous_population_within_distance_threshold": total_indigenous_population_within_threshold,
        "pct_indigenous_population_within_distance_threshold": _pct(
            total_indigenous_population_within_threshold,
            total_indigenous_population
        )
    }])

    numeric_cols = result.select_dtypes(include="number").columns
    result[numeric_cols] = result[numeric_cols].round(round_digits)

    return result

In [ ]:
access_summary_65 = summarize_db_access_coverage_from_path(
    db_table_path=r"C:\Users\AXU\OneDrive - Government of BC\Desktop\Impact Analysis\output\db_centroid_assignments.csv",
    population_year=2030,
    distance_threshold_km=15
)

access_summary_65

# Impact analysis- Turn back the four accessibility measures


In [ ]:
# ---------------------------------------------------------------------
# Define function
# Return a summarized table of scenario name, population within distance pct increase, rural population pct increase, 
# indigenous population oct increase from the analysis result 
# Inputs: analyisis result folder, population year - current is 2030  
# Outout result saved to scenario result folder 
# ---------------------------------------------------------------------


def summarize_experiment_accessibility_metrics(
    experiment_name: Union[str, Path],
    population_year: Union[int, str],
    distance_threshold_km: float,
    overall_accessibility_baseline: float,
    rural_remote_accessibility_baseline: float,
    indigenous_accessibility_baseline: float,
    output_root: Optional[Union[str, Path]] = None,
    sehi_csv_path: Optional[Union[str, Path]] = None,
    output_filename: Optional[str] = None,
    save_output: bool = True,
    distance_col: str = "centroid_distance_m",
    urban_rural_col: str = "urban_rural",
    csd_name_col: str = "csd_name",
    rural_remote_categories: Sequence[str] = ("Rural 1", "Rural 2", "Rural 3"),
    indigenous_categories: Sequence[str] = ("Indigenous",),
    sehi_csd_name_col: str = "MUN_NAME_2021",
    sehi_total_index_col: str = "TOTAL_INDEX_0_100",
    db_table_glob_patterns: Sequence[str] = (
        # "**/csb/*.csv",
        # "**/*db_centroid*.csv",
        "**/db_centroid_assignments_updated.csv",
        # "**/*db_table*.csv",
        # "**/*DB*.csv"
    ),
    round_digits: int = 2
) -> pd.DataFrame:
    """
    Summarize three accessibility metrics for each scenario in one experiment folder.

    Inputs:
      - experiment_name:
            Either an experiment folder name, for example:
                "add_13_locations_abbotsford_and_spallumcheen_and_ca_11385c55"
            or a full experiment folder path.

      - population_year:
            Projection year to use.
            Must match a DB population column such as pop_2025.

      - distance_threshold_km:
            Centroid-distance threshold in kilometres.

      - overall_accessibility_baseline:
            Overall baseline accessibility percentage.
            Expected scale: 0 to 100.

      - rural_remote_accessibility_baseline:
            Rural and remote baseline accessibility percentage.
            Expected scale: 0 to 100.

      - indigenous_accessibility_baseline:
            Indigenous baseline accessibility percentage.
            Expected scale: 0 to 100.

      - output_root:
            Root output folder, usually OUTPUT_DIRECTORY.
            Required if experiment_name is a folder name rather than a full path.

      - sehi_csv_path:
            Path to SEHI CSD-level CSV.
            If None, the function uses global SEHI_CSV.

      - output_filename:
            Optional CSV output name.
            If None, a default name is created.

      - save_output:
            If True, save result CSV into the experiment folder.

    Outputs:
      - Returns a DataFrame with:
            csd_name
            scenario_name
            overall_population_within_distance_threshold_pct
            overall_population_within_distance_threshold_pct_increase
            rural_remote_population_within_distance_threshold_pct
            rural_remote_population_within_distance_threshold_pct_increase
            indigenous_population_within_distance_threshold_pct
            indigenous_population_within_distance_threshold_pct_increase
            combined_accessibility_increase
            total_index_0_100
            weighted_avg_distance_km
    """

    # ---------------------------------------------------------------------
    # Helper: resolve experiment folder
    # ---------------------------------------------------------------------
    experiment_path = Path(experiment_name)

    if not experiment_path.exists():
        if output_root is None:
            raise ValueError(
                "output_root is required when experiment_name is not a full path."
            )

        experiment_path = Path(output_root) / str(experiment_name)

    if not experiment_path.exists():
        raise FileNotFoundError(
            f"Experiment folder does not exist: {experiment_path}"
        )

    if not experiment_path.is_dir():
        raise ValueError(
            f"experiment_name must resolve to a folder: {experiment_path}"
        )

    # ---------------------------------------------------------------------
    # Helper: resolve SEHI CSV
    # ---------------------------------------------------------------------
    if sehi_csv_path is None:
        try:
            sehi_csv_path = SEHI_CSV
        except NameError:
            raise ValueError(
                "sehi_csv_path is None, but global SEHI_CSV is not defined."
            )

    sehi_csv_path = Path(sehi_csv_path)

    if not sehi_csv_path.exists():
        raise FileNotFoundError(f"SEHI CSV does not exist: {sehi_csv_path}")

    sehi_df = pd.read_csv(sehi_csv_path, low_memory=False)

    required_sehi_cols = {
        sehi_csd_name_col,
        sehi_total_index_col
    }

    missing_sehi_cols = required_sehi_cols - set(sehi_df.columns)

    if missing_sehi_cols:
        raise ValueError(
            f"SEHI CSV missing required columns: {sorted(missing_sehi_cols)}. "
            f"Available columns: {list(sehi_df.columns)}"
        )

    # ---------------------------------------------------------------------
    # Helper: normalize names for matching scenario names to SEHI CSD names
    # ---------------------------------------------------------------------
    def _slug(value: object) -> str:
        value = "" if pd.isna(value) else str(value)
        value = value.strip().lower()
        value = re.sub(r"[^a-z0-9]+", "_", value)
        value = re.sub(r"_+", "_", value)
        return value.strip("_")

    # ---------------------------------------------------------------------
    # Helper: derive readable CSD name from scenario name
    # ---------------------------------------------------------------------
    def _derive_csd_name_from_scenario_name(value: object) -> str:
        """Derive readable CSD name when SEHI matching is unavailable."""
        if pd.isna(value):
            return ""

        text = str(value).strip()

        text = re.sub(r"^single_", "", text, flags=re.IGNORECASE)
        text = re.sub(r"^all_", "", text, flags=re.IGNORECASE)
        text = text.replace("_", " ")
        text = re.sub(r"\s+", " ", text).strip()

        return text.title()

    sehi_lookup = sehi_df.copy()
    sehi_lookup["_csd_slug"] = sehi_lookup[sehi_csd_name_col].apply(_slug)

    # ---------------------------------------------------------------------
    # Helper: find scenario DB tables
    # ---------------------------------------------------------------------
    candidate_paths = []

    for pattern in db_table_glob_patterns:
        candidate_paths.extend(experiment_path.glob(pattern))

    candidate_paths = sorted(set(candidate_paths))

    if not candidate_paths:
        raise FileNotFoundError(
            "No candidate scenario DB CSV files found under experiment folder. "
            f"Checked patterns: {list(db_table_glob_patterns)}"
        )

    population_col = f"pop_{population_year}"

    required_db_cols = {
        population_col,
        distance_col,
        urban_rural_col
    }

    # ---------------------------------------------------------------------
    # Helper: derive scenario name from file location
    # ---------------------------------------------------------------------
    def _derive_scenario_name(db_path: Path) -> str:
        rel_parts = db_path.relative_to(experiment_path).parts

        # Case 1:
        # experiment/csb/add_hope.csv
        if len(rel_parts) >= 2 and rel_parts[0].lower() == "csb":
            return db_path.stem

        # Case 2:
        # experiment/add_hope/csb/db_centroid_assignments.csv
        if len(rel_parts) >= 3 and rel_parts[1].lower() == "csb":
            return rel_parts[0]

        # Case 3:
        # experiment/add_hope/db_centroid_assignments.csv
        if len(rel_parts) >= 2:
            return rel_parts[0]

        # Case 4:
        # experiment/add_hope.csv
        return db_path.stem

    # ---------------------------------------------------------------------
    # Helper: calculate percentage
    # ---------------------------------------------------------------------
    def _pct(numerator: float, denominator: float) -> float:
        if denominator == 0 or pd.isna(denominator):
            return np.nan
        return numerator / denominator * 100

    # ---------------------------------------------------------------------
    # Helper: summarize one scenario DB table
    # ---------------------------------------------------------------------
    def _summarize_one_db_table(db_path: Path) -> Optional[dict]:
        try:
            db = pd.read_csv(db_path, low_memory=False)
        except Exception:
            return None

        missing_db_cols = required_db_cols - set(db.columns)

        # Change: Skip non-DB CSVs such as already-created summary outputs.
        if missing_db_cols:
            return None

        df = db.copy()

        df[population_col] = pd.to_numeric(
            df[population_col],
            errors="coerce"
        ).fillna(0)

        df[distance_col] = pd.to_numeric(
            df[distance_col],
            errors="coerce"
        )

        if (df[population_col] < 0).any():
            raise ValueError(
                f"{population_col} contains negative values in: {db_path}"
            )

        distance_threshold_m = distance_threshold_km * 1000

        within_threshold = (
            df[distance_col].notna()
            & (df[distance_col] <= distance_threshold_m)
        )

        urban_rural_label = (
            df[urban_rural_col]
            .astype(str)
            .str.strip()
            .str.lower()
            .str.replace(r"\s+", " ", regex=True)
        )

        rural_remote_normalized = {
            str(value).strip().lower()
            for value in rural_remote_categories
        }

        indigenous_normalized = {
            str(value).strip().lower()
            for value in indigenous_categories
        }

        is_rural_remote = urban_rural_label.isin(rural_remote_normalized)
        is_indigenous = urban_rural_label.isin(indigenous_normalized)

        total_population = float(df[population_col].sum())

        population_within = float(
            df.loc[within_threshold, population_col].sum()
        )

        rural_remote_population = float(
            df.loc[is_rural_remote, population_col].sum()
        )

        rural_remote_population_within = float(
            df.loc[
                is_rural_remote & within_threshold,
                population_col
            ].sum()
        )

        indigenous_population = float(
            df.loc[is_indigenous, population_col].sum()
        )

        indigenous_population_within = float(
            df.loc[
                is_indigenous & within_threshold,
                population_col
            ].sum()
        )

        overall_pct = _pct(
            population_within,
            total_population
        )

        rural_remote_pct = _pct(
            rural_remote_population_within,
            rural_remote_population
        )

        indigenous_pct = _pct(
            indigenous_population_within,
            indigenous_population
        )

        overall_pct_increase = (
            overall_pct - overall_accessibility_baseline
            if not pd.isna(overall_pct)
            else np.nan
        )

        rural_remote_pct_increase = (
            rural_remote_pct - rural_remote_accessibility_baseline
            if not pd.isna(rural_remote_pct)
            else np.nan
        )

        indigenous_pct_increase = (
            indigenous_pct - indigenous_accessibility_baseline
            if not pd.isna(indigenous_pct)
            else np.nan
        )

        # Change: Add combined accessibility increase as simple additive score.
        combined_accessibility_increase = (
            overall_pct_increase
            + rural_remote_pct_increase
            + indigenous_pct_increase
            if not any(
                pd.isna(value)
                for value in [
                    overall_pct_increase,
                    rural_remote_pct_increase,
                    indigenous_pct_increase
                ]
            )
            else np.nan
        )

        valid_distance = (
            df[distance_col].notna()
            & df[population_col].notna()
            & (df[population_col] > 0)
        )

        weighted_avg_distance_km = np.nan

        if df.loc[valid_distance, population_col].sum() > 0:
            weighted_avg_distance_km = (
                (
                    df.loc[valid_distance, population_col]
                    * df.loc[valid_distance, distance_col]
                    / 1000
                ).sum()
                / df.loc[valid_distance, population_col].sum()
            )

        scenario_name = _derive_scenario_name(db_path)
        scenario_slug = _slug(scenario_name)

        # -----------------------------------------------------------------
        # Match scenario name to SEHI CSD name
        # -----------------------------------------------------------------
        matched_sehi = sehi_lookup[
            sehi_lookup["_csd_slug"].apply(
                lambda csd_slug: bool(csd_slug) and csd_slug in scenario_slug
            )
        ].copy()

        if matched_sehi.empty:
            matched_csd_names = np.nan
            total_index_value = np.nan

            # Change: Add readable CSD name as first output field fallback.
            output_csd_name = _derive_csd_name_from_scenario_name(scenario_name)

        elif len(matched_sehi) == 1:
            matched_csd_names = matched_sehi.iloc[0][sehi_csd_name_col]

            total_index_value = pd.to_numeric(
                matched_sehi.iloc[0][sehi_total_index_col],
                errors="coerce"
            )

            # Change: Add matched CSD name as first output field.
            output_csd_name = str(matched_csd_names)

        else:
            # Change: Multiple CSDs can match one scenario name.
            matched_csd_names = "; ".join(
                sorted(matched_sehi[sehi_csd_name_col].astype(str).unique())
            )

            total_index_values = pd.to_numeric(
                matched_sehi[sehi_total_index_col],
                errors="coerce"
            ).dropna()

            # Change: Use mean index for multi-CSD scenarios to keep one row.
            total_index_value = (
                float(total_index_values.mean())
                if not total_index_values.empty
                else np.nan
            )

            # Change: Add matched CSD names as first output field.
            output_csd_name = matched_csd_names

        return {
            # Change: Add CSD name as first logical output column.
            "csd_name": output_csd_name,

            "scenario_name": scenario_name,
            "source_db_table": str(db_path),

            "population_year": int(population_year),
            "distance_threshold_km": distance_threshold_km,

            "total_population": total_population,
            "total_population_within_distance_threshold": population_within,
            "overall_population_within_distance_threshold_pct": overall_pct,
            "overall_population_within_distance_threshold_pct_increase": (
                overall_pct_increase
            ),

            "rural_remote_population": rural_remote_population,
            "rural_remote_population_within_distance_threshold": (
                rural_remote_population_within
            ),
            "rural_remote_population_within_distance_threshold_pct": (
                rural_remote_pct
            ),
            "rural_remote_population_within_distance_threshold_pct_increase": (
                rural_remote_pct_increase
            ),

            "indigenous_population": indigenous_population,
            "indigenous_population_within_distance_threshold": (
                indigenous_population_within
            ),
            "indigenous_population_within_distance_threshold_pct": (
                indigenous_pct
            ),
            "indigenous_population_within_distance_threshold_pct_increase": (
                indigenous_pct_increase
            ),

            "combined_accessibility_increase": combined_accessibility_increase,

            "matched_csd_names_from_sehi": matched_csd_names,
            "total_index_0_100": total_index_value,
            "weighted_avg_distance_km": weighted_avg_distance_km
        }

    # ---------------------------------------------------------------------
    # Summarize all scenario DB tables
    # ---------------------------------------------------------------------
    records = []

    for db_path in candidate_paths:
        record = _summarize_one_db_table(db_path)

        if record is not None:
            records.append(record)

    if not records:
        raise ValueError(
            "Candidate CSVs were found, but none contained the required DB columns: "
            f"{sorted(required_db_cols)}"
        )

    result = pd.DataFrame(records)

    # ---------------------------------------------------------------------
    # Change: Raise error if duplicate scenario names remain.
    # This prevents silent, unstable results from multiple DB files per scenario.
    # ---------------------------------------------------------------------
    duplicate_scenarios = result[
        result["scenario_name"].duplicated(keep=False)
    ].sort_values(["scenario_name", "source_db_table"])

    if not duplicate_scenarios.empty:
        raise ValueError(
            "Duplicate scenario_name values found. "
            "Multiple DB files are being summarized for the same scenario. "
            "Restrict db_table_glob_patterns or clean the experiment folder.\n\n"
            f"{duplicate_scenarios[['scenario_name', 'source_db_table']].to_string(index=False)}"
        )

    # ---------------------------------------------------------------------
    # Reorder and round output columns
    # ---------------------------------------------------------------------
    front_cols = [
        # Change: Put CSD name as the first output column.
        "csd_name",

        "scenario_name",

        "overall_population_within_distance_threshold_pct",
        "overall_population_within_distance_threshold_pct_increase",

        "rural_remote_population_within_distance_threshold_pct",
        "rural_remote_population_within_distance_threshold_pct_increase",

        "indigenous_population_within_distance_threshold_pct",
        "indigenous_population_within_distance_threshold_pct_increase",

        "combined_accessibility_increase",

        "total_index_0_100",
        "weighted_avg_distance_km"
    ]

    remaining_cols = [
        col for col in result.columns
        if col not in front_cols
    ]

    result = result[front_cols + remaining_cols]

    numeric_cols = result.select_dtypes(include="number").columns
    result[numeric_cols] = result[numeric_cols].round(round_digits)

    result = (
        result
        .sort_values(
            [
                "combined_accessibility_increase",
                "overall_population_within_distance_threshold_pct_increase",
                "rural_remote_population_within_distance_threshold_pct_increase",
                "indigenous_population_within_distance_threshold_pct_increase",
                "weighted_avg_distance_km"
            ],
            ascending=[False, False, False, False, True]
        )
        .reset_index(drop=True)
    )

    # ---------------------------------------------------------------------
    # Save output
    # ---------------------------------------------------------------------
    if save_output:
        if output_filename is None:
            output_filename = (
                f"scenario_accessibility_summary_"
                f"pop_{population_year}_"
                f"{distance_threshold_km:g}km.csv"
            )

        output_path = experiment_path / output_filename
        result.to_csv(output_path, index=False)

    return result

In [ ]:
# Run scenario test for adding 13 locations. 
scenario_access_impact_78 = summarize_experiment_accessibility_metrics(
    experiment_name="add_13_locations_spallumcheen_and_castleg_0f620216_run_003",
    output_root=OUTPUT_DIRECTORY,
    population_year=2030,
    distance_threshold_km=15,
    overall_accessibility_baseline=82.2,
    rural_remote_accessibility_baseline=58.23,
    indigenous_accessibility_baseline=66.59,
    sehi_csv_path=SEHI_CSV,
    sehi_csd_name_col="MUN_NAME_2021",
    sehi_total_index_col="TOTAL_INDEX_0_100",
    save_output=True

)

scenario_access_impact_78

# Priority Index table 

In [ ]:
# ---------------------------------------------------------------------
# Define function
# Rank the locations by giving equal weight on these four factors:
# overall population pct increase, rural population pct increase indigenous population pct increase and SEHI priority
# ---------------------------------------------------------------------


def build_weighted_accessibility_priority_table(
    scenario_summary: pd.DataFrame,
    w1: float,
    w2: float,
    w3: float,
    w4: float,
    total_index_threshold: float = 50,
    csd_name_col: str = "csd_name",
    scenario_name_col: str = "scenario_name",
    overall_increase_col: str = "overall_population_within_distance_threshold_pct_increase",
    rural_remote_increase_col: str = "rural_remote_population_within_distance_threshold_pct_increase",
    indigenous_increase_col: str = "indigenous_population_within_distance_threshold_pct_increase",
    total_index_col: str = "total_index_0_100",
    weighted_avg_distance_col: str = "weighted_avg_distance_km",
    source_db_table_col: str = "source_db_table",
    clip_negative_increases: bool = True,
    normalize_weights: bool = False,
    experiment_name: Optional[Union[str, Path]] = None,
    output_root: Optional[Union[str, Path]] = None,
    output_filename: Optional[str] = None,
    save_output: bool = True,
    round_digits: int = 4
) -> pd.DataFrame:
    """
    Build a weighted accessibility priority score table from
    summarize_experiment_accessibility_metrics() output and optionally save it
    to the experiment folder.

    Output:
      - csd_name
      - scenario_name
      - raw accessibility increase columns
      - total_index_0_100
      - weighted_avg_distance_km
      - SEHI_priority
      - normalized priority component columns
      - weighted priority_score

    Save logic:
      - If experiment_name is provided, save to that experiment folder.
      - If experiment_name is not provided, infer experiment folder from source_db_table.
    """

    # ---------------------------------------------------------------------
    # Validate required columns
    # ---------------------------------------------------------------------
    required_cols = {
        csd_name_col,
        scenario_name_col,
        overall_increase_col,
        rural_remote_increase_col,
        indigenous_increase_col,
        total_index_col,
        weighted_avg_distance_col
    }

    missing_cols = required_cols - set(scenario_summary.columns)

    if missing_cols:
        raise ValueError(
            f"scenario_summary missing required columns: {sorted(missing_cols)}"
        )

    df = scenario_summary.copy()

    # ---------------------------------------------------------------------
    # Validate and optionally normalize weights
    # ---------------------------------------------------------------------
    weights = np.array([w1, w2, w3, w4], dtype=float)

    if np.isnan(weights).any():
        raise ValueError("Weights cannot contain NaN.")

    if (weights < 0).any():
        raise ValueError("Weights should be non-negative.")

    if normalize_weights:
        weight_sum = weights.sum()

        if weight_sum == 0:
            raise ValueError(
                "Cannot normalize weights because all weights are zero."
            )

        weights = weights / weight_sum

    w1, w2, w3, w4 = weights.tolist()

    # ---------------------------------------------------------------------
    # Clean CSD and scenario names
    # ---------------------------------------------------------------------
    df[csd_name_col] = (
        df[csd_name_col]
        .astype("string")
        .fillna("")
        .str.strip()
    )

    df[csd_name_col] = df[csd_name_col].mask(
        df[csd_name_col].str.lower().isin(["nan", "none", "null"]),
        ""
    )

    df[scenario_name_col] = (
        df[scenario_name_col]
        .astype("string")
        .fillna("")
        .str.strip()
    )

    # ---------------------------------------------------------------------
    # Convert metric columns to numeric
    # ---------------------------------------------------------------------
    numeric_cols = [
        overall_increase_col,
        rural_remote_increase_col,
        indigenous_increase_col,
        total_index_col,
        weighted_avg_distance_col
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # ---------------------------------------------------------------------
    # Build SEHI priority
    # Excel equivalent: =IF(F2="",0,MAX(50-F2,0))
    # ---------------------------------------------------------------------
    df["SEHI_priority"] = np.where(
        df[total_index_col].isna(),
        0,
        np.maximum(total_index_threshold - df[total_index_col], 0)
    )

    # ---------------------------------------------------------------------
    # Prepare values for normalization
    # ---------------------------------------------------------------------
    component_input_cols = {
        overall_increase_col: "overall_accessibility_increase_input",
        rural_remote_increase_col: "rural_remote_accessibility_increase_input",
        indigenous_increase_col: "indigenous_accessibility_increase_input",
        "SEHI_priority": "SEHI_priority_input"
    }

    for source_col, input_col in component_input_cols.items():
        df[input_col] = pd.to_numeric(
            df[source_col],
            errors="coerce"
        ).fillna(0)

        # Change: Negative increases should not improve priority ranking.
        if clip_negative_increases and source_col != "SEHI_priority":
            df[input_col] = df[input_col].clip(lower=0)

    # ---------------------------------------------------------------------
    # Normalize component columns using min-max normalization
    # ---------------------------------------------------------------------
    def _min_max_normalize(series: pd.Series) -> pd.Series:
        series = pd.to_numeric(series, errors="coerce").fillna(0)

        min_value = series.min()
        max_value = series.max()

        if pd.isna(min_value) or pd.isna(max_value) or max_value == min_value:
            return pd.Series(0.0, index=series.index)

        return (series - min_value) / (max_value - min_value)

    df["overall_population_within_distance_threshold_pct_increase_norm"] = (
        _min_max_normalize(df["overall_accessibility_increase_input"])
    )

    df["rural_remote_population_within_distance_threshold_pct_increase_norm"] = (
        _min_max_normalize(df["rural_remote_accessibility_increase_input"])
    )

    df["indigenous_population_within_distance_threshold_pct_increase_norm"] = (
        _min_max_normalize(df["indigenous_accessibility_increase_input"])
    )

    df["SEHI_priority_norm"] = (
        _min_max_normalize(df["SEHI_priority_input"])
    )

    # ---------------------------------------------------------------------
    # Calculate weighted priority score
    # ---------------------------------------------------------------------
    df["priority_score"] = (
        w1 * df["overall_population_within_distance_threshold_pct_increase_norm"]
        + w2 * df["rural_remote_population_within_distance_threshold_pct_increase_norm"]
        + w3 * df["indigenous_population_within_distance_threshold_pct_increase_norm"]
        + w4 * df["SEHI_priority_norm"]
    )

    # ---------------------------------------------------------------------
    # Sort and add priority rank
    # ---------------------------------------------------------------------
    result = (
        df
        .sort_values(
            [
                "priority_score",
                overall_increase_col,
                rural_remote_increase_col,
                indigenous_increase_col,
                "SEHI_priority",
                weighted_avg_distance_col
            ],
            ascending=[False, False, False, False, False, True]
        )
        .reset_index(drop=True)
    )

    result["priority_rank"] = np.arange(1, len(result) + 1)

    # ---------------------------------------------------------------------
    # Reorder output columns
    # ---------------------------------------------------------------------
    output_cols = [
        "priority_rank",
        csd_name_col,
        scenario_name_col,

        overall_increase_col,
        rural_remote_increase_col,
        indigenous_increase_col,

        total_index_col,
        weighted_avg_distance_col,
        "SEHI_priority",

        "overall_population_within_distance_threshold_pct_increase_norm",
        "rural_remote_population_within_distance_threshold_pct_increase_norm",
        "indigenous_population_within_distance_threshold_pct_increase_norm",
        "SEHI_priority_norm",

        "priority_score"
    ]

    result = result[output_cols].copy()

    # ---------------------------------------------------------------------
    # Round numeric output
    # ---------------------------------------------------------------------
    numeric_result_cols = result.select_dtypes(include="number").columns
    result[numeric_result_cols] = result[numeric_result_cols].round(round_digits)

    # ---------------------------------------------------------------------
    # Change: Save output to experiment folder
    # ---------------------------------------------------------------------
    def _resolve_experiment_path() -> Path:
        if experiment_name is not None:
            experiment_path = Path(experiment_name)

            if not experiment_path.exists():
                if output_root is None:
                    raise ValueError(
                        "output_root is required when experiment_name is not a full path."
                    )

                experiment_path = Path(output_root) / str(experiment_name)

            if not experiment_path.exists():
                raise FileNotFoundError(
                    f"Experiment folder does not exist: {experiment_path}"
                )

            if not experiment_path.is_dir():
                raise ValueError(
                    f"experiment_name must resolve to a folder: {experiment_path}"
                )

            return experiment_path

        # Change: Infer experiment folder from source_db_table if available.
        if source_db_table_col not in scenario_summary.columns:
            raise ValueError(
                "Cannot infer experiment folder because source_db_table is missing. "
                "Pass experiment_name and output_root explicitly."
            )

        source_paths = (
            scenario_summary[source_db_table_col]
            .dropna()
            .astype(str)
            .str.strip()
        )

        source_paths = source_paths[source_paths != ""]

        if source_paths.empty:
            raise ValueError(
                "Cannot infer experiment folder because source_db_table is empty. "
                "Pass experiment_name and output_root explicitly."
            )

        inferred_paths = []

        for source_path in source_paths:
            path = Path(source_path)

            # Expected:
            # experiment_folder / scenario_folder / db_centroid_assignments_updated.csv
            inferred_paths.append(path.parent.parent)

        unique_inferred_paths = sorted({
            str(path)
            for path in inferred_paths
        })

        if len(unique_inferred_paths) != 1:
            raise ValueError(
                "Cannot infer a single experiment folder. "
                f"Found multiple inferred folders: {unique_inferred_paths}"
            )

        experiment_path = Path(unique_inferred_paths[0])

        if not experiment_path.exists():
            raise FileNotFoundError(
                f"Inferred experiment folder does not exist: {experiment_path}"
            )

        return experiment_path

    if save_output:
        experiment_path = _resolve_experiment_path()

        if output_filename is None:
            output_filename = "weighted_accessibility_priority_table.csv"

        output_path = experiment_path / output_filename

        result.to_csv(output_path, index=False)

        # Change: Store saved path in DataFrame metadata.
        result.attrs["output_path"] = str(output_path)

    return result

In [ ]:
# Generate priority locations output 
priority_table = build_weighted_accessibility_priority_table(
    scenario_summary=scenario_access_impact_78,
    w1=0.25,
    w2=0.25,
    w3=0.25,
    w4=0.25,
    total_index_threshold=50,
    clip_negative_increases=True,
    normalize_weights=False
)

priority_table

# Other supporting analysis: Summary DB to CSD

In [ ]:
# -----------------------------------------------------------------------
# Function: summarize_csd_nearest_office_distribution
#
# Description: Summarize DB-level nearest-office assignment results to the
#              CSD level and return the result directly as a DataFrame.
#              For each CSD, the function calculates:
#                   - total projected population for the selected year
#                   - population-weighted average centroid distance to the
#                     assigned nearest SBC office
#                   - SEHI total index score from the SEHI CSD-level table
#                   - number of distinct nearest offices assigned to DBs
#                     within the CSD
#                   - dynamic office/population columns showing how much
#                     CSD population is assigned to each nearest office
#
# Inputs:
#   - sehi_csv_path: path to SEHI CSD-level CSV.
#                    Required columns after clean_names():
#                       csd_uid
#                       total_index_0_100
#                    Optional name columns:
#                       mun_nam, mun_name, csd_name, municipality_name, name
#   - csv1_path: path to DB centroid assignment CSV.
#                Expected file is usually:
#                       db_centroid_assignments.csv
#                    or
#                       db_centroid_assignments_updated.csv
#                Required columns after clean_names():
#                       csdid
#                       csd_name
#                       pop_YYYY
#                       assigned_facility
#                       centroid_distance_m
#   - population_year: projection year to use, e.g. 2025, 2030, 2035.
#                      Must correspond to a column such as pop_2025.
#
# Outputs:
#   - Returns a pandas DataFrame with one row per CSD:
#       - csd_id
#       - csd_name
#       - population_total
#       - weighted_avg_distance_m
#       - weighted_avg_distance_km
#       - total_index_0_100
#       - nearest_office_count
#       - nearest_office_1
#       - nearest_office_1_population
#       - nearest_office_2
#       - nearest_office_2_population
#       - ...
#
# Assumptions:
#   - csv1_path contains one row per DB assignment.
#   - centroid_distance_m is measured in metres.
#   - centroid_distance_m is straight-line centroid distance, not drive distance.
#   - Population values that cannot be parsed are treated as zero.
#   - Rows with missing assigned_facility are excluded from office counts.
#   - Office columns are ordered by assigned population descending within each CSD.
# ------------------------------------------------------------------------

def summarize_csd_nearest_office_distribution(
    sehi_csv_path: str,
    csv1_path: str,
    population_year: int,
    rural_matrix_path: str = None,
    rural_matrix_sheet_name: str = "Census Subdivision Data"
) -> pd.DataFrame:
    """
    Summarize nearest-office assignment results from DB level to CSD level.
    Returns the final table directly.

    Added from rural matrix:
      - rural_category
      - ir_score_2021
      - ferry_access_only
    """

    # ---------------------------------------------------------------------
    # Helper: normalize CSD id values
    # ---------------------------------------------------------------------
    def normalize_csd_id(series: pd.Series) -> pd.Series:
        return (
            series
            .astype(str)
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
        )

    # ---------------------------------------------------------------------
    # Helper: pick first available column from a list
    # ---------------------------------------------------------------------
    def first_existing_col(df: pd.DataFrame, candidates: list) -> str:
        return next((col for col in candidates if col in df.columns), None)

    # ---------------------------------------------------------------------
    # Read input files
    # ---------------------------------------------------------------------
    sehi = pd.read_csv(sehi_csv_path)
    db = pd.read_csv(csv1_path)

    sehi.columns = clean_names(sehi.columns)
    db.columns = clean_names(db.columns)

    pop_col = f"pop_{population_year}"

    # ---------------------------------------------------------------------
    # Validate required SEHI columns
    # ---------------------------------------------------------------------
    required_sehi_cols = {
        "csd_uid",
        "total_index_0_100"
    }

    missing_sehi_cols = required_sehi_cols - set(sehi.columns)

    if missing_sehi_cols:
        raise ValueError(
            f"SEHI CSV missing required columns after clean_names(): "
            f"{sorted(missing_sehi_cols)}. "
            f"Available SEHI columns: {list(sehi.columns)}"
        )

    # ---------------------------------------------------------------------
    # Validate required DB assignment columns
    # ---------------------------------------------------------------------
    required_db_cols = {
        "csdid",
        "csd_name",
        pop_col,
        "assigned_facility",
        "centroid_distance_m"
    }

    missing_db_cols = required_db_cols - set(db.columns)

    if missing_db_cols:
        raise ValueError(
            f"DB assignment CSV missing required columns after clean_names(): "
            f"{sorted(missing_db_cols)}. "
            f"Available DB columns: {list(db.columns)}"
        )

    # ---------------------------------------------------------------------
    # Build SEHI reference table
    # ---------------------------------------------------------------------
    possible_sehi_name_cols = [
        "mun_nam",
        "mun_name",
        "csd_name",
        "municipality_name",
        "name"
    ]

    sehi_name_col = first_existing_col(sehi, possible_sehi_name_cols)

    sehi_cols = ["csd_uid", "total_index_0_100"]

    if sehi_name_col:
        sehi_cols.append(sehi_name_col)

    sehi_ref = (
        sehi[sehi_cols]
        .drop_duplicates()
        .rename(columns={"csd_uid": "csd_id"})
    )

    if sehi_name_col:
        sehi_ref = sehi_ref.rename(columns={sehi_name_col: "csd_name_sehi"})
    else:
        sehi_ref["csd_name_sehi"] = None

    sehi_ref["csd_id"] = normalize_csd_id(sehi_ref["csd_id"])

    sehi_ref["total_index_0_100"] = pd.to_numeric(
        sehi_ref["total_index_0_100"],
        errors="coerce"
    )

    # ---------------------------------------------------------------------
    # Build rural matrix reference table
    # ---------------------------------------------------------------------
    if rural_matrix_path is None:
        if "RURAL_MATRIX_XLSX" not in globals():
            raise ValueError(
                "rural_matrix_path is None and global RURAL_MATRIX_XLSX is not defined."
            )
        rural_matrix_path = RURAL_MATRIX_XLSX

    rural = pd.read_excel(
        rural_matrix_path,
        sheet_name=rural_matrix_sheet_name
    )

    rural.columns = clean_names(rural.columns)

    rural_csd_id_col = first_existing_col(
        rural,
        [
            "csduid",
            "csd_uid",
            "csd_id",
            "census_subdivision_id",
            "census_subdivision_uid"
        ]
    )

    if rural_csd_id_col is None:
        raise ValueError(
            "Rural matrix missing CSD id column. Expected one of: "
            "csduid, csd_uid, csd_id, census_subdivision_id, census_subdivision_uid. "
            f"Available rural matrix columns: {list(rural.columns)}"
        )

    rural_category_col = first_existing_col(
        rural,
        [
            "rural_category",
            "rural_urban",
            "urban_rural",
            "rurality",
            "ri_rural_category"
        ]
    )

    if rural_category_col is None:
        raise ValueError(
            "Rural matrix missing rural category column. Expected one of: "
            "rural_category, rural_urban, urban_rural, rurality, ri_rural_category. "
            f"Available rural matrix columns: {list(rural.columns)}"
        )

    ir_score_col = first_existing_col(
        rural,
        [
            "ir_score_2021",
            "ir_2021",
            "ir_score",
            "ir_score_21",
            "index_of_remoteness_score_2021",
            "index_of_remoteness_2021",
            "index_of_remoteness_score",
            "remoteness_score_2021",
            "remoteness_score"
        ]
    )

    if ir_score_col is None:
        raise ValueError(
            "Rural matrix missing IR score 2021 column. Expected one of: "
            "ir_score_2021, ir_2021, ir_score, index_of_remoteness_score_2021, "
            "index_of_remoteness_2021, remoteness_score_2021. "
            f"Available rural matrix columns: {list(rural.columns)}"
        )

    # Change: Add ferry-access-only field from rural matrix.
    ferry_access_only_col = first_existing_col(
        rural,
        [
            "ferry_access_only",
            "ferry_access",
            "ferry_only",
            "ferry_access_only_community",
            "ferry_access_only_flag",
            "ferry_dependent",
            "ferry_dependent_community"
        ]
    )

    if ferry_access_only_col is None:
        raise ValueError(
            "Rural matrix missing ferry access only column. Expected one of: "
            "ferry_access_only, ferry_access, ferry_only, "
            "ferry_access_only_community, ferry_access_only_flag, "
            "ferry_dependent, ferry_dependent_community. "
            f"Available rural matrix columns: {list(rural.columns)}"
        )

    rural_ref = (
        rural[
            [
                rural_csd_id_col,
                rural_category_col,
                ir_score_col,
                ferry_access_only_col
            ]
        ]
        .drop_duplicates()
        .rename(columns={
            rural_csd_id_col: "csd_id",
            rural_category_col: "rural_category",
            ir_score_col: "ir_score_2021",
            ferry_access_only_col: "ferry_access_only"
        })
    )

    rural_ref["csd_id"] = normalize_csd_id(rural_ref["csd_id"])

    rural_ref["rural_category"] = (
        rural_ref["rural_category"]
        .astype(str)
        .str.strip()
        .replace({"nan": None, "None": None})
    )

    rural_ref["ir_score_2021"] = pd.to_numeric(
        rural_ref["ir_score_2021"],
        errors="coerce"
    )

    # Change: Standardize ferry_access_only as text/flag without forcing boolean.
    rural_ref["ferry_access_only"] = (
        rural_ref["ferry_access_only"]
        .astype(str)
        .str.strip()
        .replace({"nan": None, "None": None})
    )

    # Change: Keep one rural-matrix record per CSD id to prevent row multiplication.
    rural_ref = rural_ref.drop_duplicates(subset=["csd_id"], keep="first")

    # ---------------------------------------------------------------------
    # Standardize DB assignment table
    # ---------------------------------------------------------------------
    db["csdid"] = normalize_csd_id(db["csdid"])

    db[pop_col] = pd.to_numeric(
        db[pop_col],
        errors="coerce"
    ).fillna(0)

    db["centroid_distance_m"] = pd.to_numeric(
        db["centroid_distance_m"],
        errors="coerce"
    )

    # ---------------------------------------------------------------------
    # Calculate CSD-level population and weighted average distance
    # ---------------------------------------------------------------------
    db["weighted_distance_component"] = (
        db[pop_col] * db["centroid_distance_m"]
    )

    csd_base = (
        db.groupby(["csdid", "csd_name"], as_index=False)
        .agg(
            population_total=(pop_col, "sum"),
            weighted_distance_numerator=("weighted_distance_component", "sum")
        )
    )

    csd_base["weighted_avg_distance_m"] = csd_base.apply(
        lambda r: (
            r["weighted_distance_numerator"] / r["population_total"]
            if r["population_total"] > 0
            else float("nan")
        ),
        axis=1
    )

    csd_base["weighted_avg_distance_km"] = (
        csd_base["weighted_avg_distance_m"] / 1000
    )

    csd_base = csd_base.drop(columns=["weighted_distance_numerator"])

    # ---------------------------------------------------------------------
    # Calculate population assigned to each nearest office within each CSD
    # ---------------------------------------------------------------------
    office_pop = (
        db[db["assigned_facility"].notna()]
        .groupby(["csdid", "assigned_facility"], as_index=False)
        .agg(office_population=(pop_col, "sum"))
    )

    office_pop = office_pop.sort_values(
        ["csdid", "office_population", "assigned_facility"],
        ascending=[True, False, True]
    )

    office_pop["office_rank"] = (
        office_pop.groupby("csdid").cumcount() + 1
    )

    # ---------------------------------------------------------------------
    # Count distinct nearest offices per CSD
    # ---------------------------------------------------------------------
    office_count = (
        office_pop.groupby("csdid", as_index=False)
        .agg(nearest_office_count=("assigned_facility", "nunique"))
    )

    # ---------------------------------------------------------------------
    # Pivot nearest office names to wide format
    # ---------------------------------------------------------------------
    office_name_wide = office_pop.pivot(
        index="csdid",
        columns="office_rank",
        values="assigned_facility"
    )

    office_name_wide.columns = [
        f"nearest_office_{int(c)}"
        for c in office_name_wide.columns
    ]

    # ---------------------------------------------------------------------
    # Pivot nearest office population to wide format
    # ---------------------------------------------------------------------
    office_pop_wide = office_pop.pivot(
        index="csdid",
        columns="office_rank",
        values="office_population"
    )

    office_pop_wide.columns = [
        f"nearest_office_{int(c)}_population"
        for c in office_pop_wide.columns
    ]

    office_wide = (
        office_name_wide
        .join(office_pop_wide, how="outer")
        .reset_index()
    )

    # ---------------------------------------------------------------------
    # Combine CSD metrics, office distribution, SEHI, and rural matrix fields
    # ---------------------------------------------------------------------
    result = (
        csd_base
        .rename(columns={"csdid": "csd_id"})
        .merge(
            office_count.rename(columns={"csdid": "csd_id"}),
            on="csd_id",
            how="left"
        )
        .merge(
            office_wide.rename(columns={"csdid": "csd_id"}),
            on="csd_id",
            how="left"
        )
        .merge(
            sehi_ref,
            on="csd_id",
            how="left"
        )
        .merge(
            rural_ref,
            on="csd_id",
            how="left"
        )
    )

    result["csd_name"] = result["csd_name"].fillna(result["csd_name_sehi"])

    result["nearest_office_count"] = (
        result["nearest_office_count"]
        .fillna(0)
        .astype(int)
    )

    # ---------------------------------------------------------------------
    # Reorder columns
    # ---------------------------------------------------------------------
    fixed_cols = [
        "csd_id",
        "csd_name",
        "population_total",
        "weighted_avg_distance_m",
        "weighted_avg_distance_km",
        "total_index_0_100",
        "rural_category",
        "ir_score_2021",
        "ferry_access_only",
        "nearest_office_count"
    ]

    office_cols = []

    max_rank = office_pop["office_rank"].max() if not office_pop.empty else 0

    for i in range(1, int(max_rank) + 1):
        office_col = f"nearest_office_{i}"
        pop_office_col = f"nearest_office_{i}_population"

        if office_col in result.columns:
            office_cols.append(office_col)

        if pop_office_col in result.columns:
            office_cols.append(pop_office_col)

    result = result[fixed_cols + office_cols]

    # ---------------------------------------------------------------------
    # Round numeric outputs
    # ---------------------------------------------------------------------
    result["population_total"] = result["population_total"].round(0)
    result["weighted_avg_distance_m"] = result["weighted_avg_distance_m"].round(1)
    result["weighted_avg_distance_km"] = result["weighted_avg_distance_km"].round(2)
    result["total_index_0_100"] = result["total_index_0_100"].round(2)
    result["ir_score_2021"] = result["ir_score_2021"].round(2)

    for col in result.columns:
        if col.endswith("_population"):
            result[col] = pd.to_numeric(result[col], errors="coerce").round(1)

    # ---------------------------------------------------------------------
    # Sort result and return directly
    # ---------------------------------------------------------------------
    result = (
        result
        .sort_values(["csd_name", "csd_id"])
        .reset_index(drop=True)
    )

    return result

In [ ]:
#CSD summary table based on 78 location 
csd_office_summary_78 = summarize_csd_nearest_office_distribution(
    sehi_csv_path=SEHI_CSV,
    csv1_path=os.path.join(
    OUTPUT_DIRECTORY,
    "add_13_locations_spallumcheen_and_castleg_0f620216_run_003",
    "all_new_locations",
    "db_centroid_assignments_updated.csv"
),
    population_year=2030
)

csd_office_summary_78.head()

# Summary table- Filter and group adjacent CSDs meeting SEHI and travel threshold 

In [ ]:
# Change: Request CSD Geometry boundary as GeoDataFrame.
csd_gdf = bcdata.get_data(
    "census-profiles-for-bc-census-subdivisions-2021-census",
    as_gdf=True
)

# Change: Confirm object type before using .columns.
print(type(csd_gdf))

if not isinstance(csd_gdf, gpd.GeoDataFrame):
    raise TypeError(
        f"Expected GeoDataFrame, but got {type(csd_gdf)}. "
        f"Object preview: {str(csd_gdf)[:500]}"
    )

# Change: Clean column names to match project convention.
csd_gdf.columns = clean_names(csd_gdf.columns)

# print(csd_gdf.columns.tolist())

# Change: Standardize CSD id column for adjacency function.
if "csd_uid" not in csd_gdf.columns:
    if "census_subdivision_uid" in csd_gdf.columns:
        csd_gdf["csd_uid"] = csd_gdf["census_subdivision_uid"]
    elif "census_subdivision_id" in csd_gdf.columns:
        csd_gdf["csd_uid"] = csd_gdf["census_subdivision_id"]
    elif "csd_id" in csd_gdf.columns:
        csd_gdf["csd_uid"] = csd_gdf["csd_id"]
    else:
        raise ValueError(
            f"Could not find CSD id column. Available columns: {list(csd_gdf.columns)}"
        )

# csd_gdf[["csd_uid", "geometry"]].head()

In [ ]:
# -----------------------------------------------------------------------
# Function: summarize_csd_nearest_office_distribution
#
# Description: Summarize CSD level adjacent other CSDs and output their populations, SEHI, weighted distances, rural index, etc. 
#
# Inputs:
#   - csd_office_summary
#
#   - csd_gdf: df of CSD Geometry boundary as GeoDataFrame from BC data
#   - sehi_threshold
#   - travel_distance_threshold_km
#    csd_summary_id_col 
#    csd_gdf_id_col: standardized CSD IDs 

# Outputs:
#   - Returns a pandas DataFrame with one row per CSD:
#       - Adjacent group 
#       - Adjacent group size
#       - Adjacent group population total  
#       - Adjacent group CSD names  
#       - csd_id
#       - csd_name
#       - population_total
#       - weighted_avg_distance_m
#       - weighted_avg_distance_km
#       - total_index_0_100
#       - Rural Index 
#       - IR score
#       - Nearest office 1, 2, 3 ...
#       - Nearest office count    
#       - Nearest office population 
# Oytput file as low_sehi_high_distance_groups saved in output folder. 
# ------------------------------------------------------------------------

def filter_and_group_adjacent_low_sehi_high_distance_csds(
    csd_office_summary: pd.DataFrame,
    csd_gdf: gpd.GeoDataFrame,
    sehi_threshold: float,
    travel_distance_threshold_km: float,
    population_total_threshold: Optional[float] = None,
    csd_summary_id_col: str = "csd_id",
    csd_gdf_id_col: str = "csdid",
    csd_name_col: str = "csd_name",
    population_col: str = "population_total",
    sehi_col: str = "total_index_0_100",
    travel_distance_col: str = "weighted_avg_distance_km",
    include_null_sehi: bool = False
) -> pd.DataFrame:
    """
    Filter CSDs with low SEHI and high travel distance, then group adjacent CSDs.

    Inputs:
      - csd_office_summary:
            DataFrame with one row per CSD.
            Required columns by default:
                csd_id
                csd_name
                population_total
                total_index_0_100
                weighted_avg_distance_km

      - csd_gdf:
            GeoDataFrame containing CSD polygon geometries.
            Required columns:
                csdid or another CSD id column
                geometry

      - sehi_threshold:
            Maximum SEHI / total index threshold.
            CSDs with SEHI < threshold are selected.

      - travel_distance_threshold_km:
            Minimum travel distance threshold.
            CSDs with weighted_avg_distance_km > threshold are selected.

      - population_total_threshold:
            Optional minimum population threshold.
            If None, no population filter is applied.

      - include_null_sehi:
            If True, CSDs with null SEHI are included as priority communities.
            If False, null SEHI is excluded.

    Output:
      - pandas DataFrame containing filtered CSDs with:
            adjacency_group_id
            adjacency_group_size
            adjacency_group_population_total
            adjacency_group_csd_names
            adjacency_group_weighted_avg_distance_km
            adjacency_group_min_sehi
            adjacency_group_max_sehi
    """

    # ---------------------------------------------------------------------
    # Helper: normalize CSD id values
    # ---------------------------------------------------------------------
    def normalize_csd_id(series: pd.Series) -> pd.Series:
        return (
            series
            .astype(str)
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
        )

    # ---------------------------------------------------------------------
    # Validate required columns
    # ---------------------------------------------------------------------
    required_summary_cols = {
        csd_summary_id_col,
        csd_name_col,
        population_col,
        sehi_col,
        travel_distance_col
    }

    missing_summary_cols = required_summary_cols - set(csd_office_summary.columns)

    if missing_summary_cols:
        raise ValueError(
            f"csd_office_summary missing required columns: "
            f"{sorted(missing_summary_cols)}"
        )

    required_gdf_cols = {csd_gdf_id_col, "geometry"}
    missing_gdf_cols = required_gdf_cols - set(csd_gdf.columns)

    if missing_gdf_cols:
        raise ValueError(
            f"csd_gdf missing required columns: {sorted(missing_gdf_cols)}"
        )

    # ---------------------------------------------------------------------
    # Prepare summary table
    # ---------------------------------------------------------------------
    summary = csd_office_summary.copy()

    summary[csd_summary_id_col] = normalize_csd_id(summary[csd_summary_id_col])

    summary[population_col] = pd.to_numeric(
        summary[population_col],
        errors="coerce"
    ).fillna(0)

    summary[sehi_col] = pd.to_numeric(
        summary[sehi_col],
        errors="coerce"
    )

    summary[travel_distance_col] = pd.to_numeric(
        summary[travel_distance_col],
        errors="coerce"
    )

    if (summary[population_col] < 0).any():
        raise ValueError(f"{population_col} contains negative values.")

    # ---------------------------------------------------------------------
    # Filter rows meeting SEHI and travel-distance thresholds
    # ---------------------------------------------------------------------
    # Change: Select low-SEHI communities.
    if include_null_sehi:
        sehi_filter = (
            (summary[sehi_col] < sehi_threshold) |
            (summary[sehi_col].isna())
        )
    else:
        sehi_filter = summary[sehi_col] < sehi_threshold

    # Change: Select high-distance communities needing access improvement.
    distance_filter = summary[travel_distance_col] > travel_distance_threshold_km

    filtered = summary[
        sehi_filter & distance_filter
    ].copy()

    # Change: Optional population filter only, not required by default.
    if population_total_threshold is not None:
        filtered = filtered[
            filtered[population_col] >= population_total_threshold
        ].copy()

    if filtered.empty:
        return filtered.assign(
            adjacency_group_id=pd.Series(dtype="Int64"),
            adjacency_group_size=pd.Series(dtype="Int64"),
            adjacency_group_population_total=pd.Series(dtype="float"),
            adjacency_group_csd_names=pd.Series(dtype="object"),
            adjacency_group_weighted_avg_distance_km=pd.Series(dtype="float"),
            adjacency_group_min_sehi=pd.Series(dtype="float"),
            adjacency_group_max_sehi=pd.Series(dtype="float")
        )

    # ---------------------------------------------------------------------
    # Join filtered rows to CSD geometry
    # ---------------------------------------------------------------------
    csd_geom = csd_gdf[[csd_gdf_id_col, "geometry"]].copy()
    csd_geom[csd_gdf_id_col] = normalize_csd_id(csd_geom[csd_gdf_id_col])

    filtered_gdf = filtered.merge(
        csd_geom,
        left_on=csd_summary_id_col,
        right_on=csd_gdf_id_col,
        how="left"
    )

    missing_geom = filtered_gdf[filtered_gdf["geometry"].isna()]

    if not missing_geom.empty:
        raise ValueError(
            "Some filtered CSDs could not be matched to geometry. "
            f"Missing CSD ids: {missing_geom[csd_summary_id_col].tolist()}"
        )

    filtered_gdf = gpd.GeoDataFrame(
        filtered_gdf,
        geometry="geometry",
        crs=csd_gdf.crs
    )

    # Change: Clean invalid geometries before adjacency check.
    filtered_gdf["geometry"] = filtered_gdf["geometry"].buffer(0)

    # ---------------------------------------------------------------------
    # Build adjacency pairs among filtered CSDs
    # ---------------------------------------------------------------------
    left = filtered_gdf[
        [csd_summary_id_col, csd_name_col, "geometry"]
    ].rename(columns={
        csd_summary_id_col: "csd_id_left",
        csd_name_col: "csd_name_left"
    })

    right = filtered_gdf[
        [csd_summary_id_col, csd_name_col, "geometry"]
    ].rename(columns={
        csd_summary_id_col: "csd_id_right",
        csd_name_col: "csd_name_right"
    })

    left = gpd.GeoDataFrame(left, geometry="geometry", crs=filtered_gdf.crs)
    right = gpd.GeoDataFrame(right, geometry="geometry", crs=filtered_gdf.crs)

    # Change: predicate="touches" groups CSDs that share a boundary.
    adjacency_pairs = gpd.sjoin(
        left,
        right,
        how="inner",
        predicate="touches"
    )

    # Remove self-pairs.
    adjacency_pairs = adjacency_pairs[
        adjacency_pairs["csd_id_left"] != adjacency_pairs["csd_id_right"]
    ]

    # ---------------------------------------------------------------------
    # Build connected adjacency groups
    # ---------------------------------------------------------------------
    csd_ids = filtered_gdf[csd_summary_id_col].tolist()

    adjacency_map = {csd_id: set() for csd_id in csd_ids}

    for _, row in adjacency_pairs.iterrows():
        left_id = row["csd_id_left"]
        right_id = row["csd_id_right"]

        adjacency_map[left_id].add(right_id)
        adjacency_map[right_id].add(left_id)

    visited = set()
    group_records = []
    group_id = 1

    for csd_id in csd_ids:
        if csd_id in visited:
            continue

        stack = [csd_id]
        group_members = []

        while stack:
            current = stack.pop()

            if current in visited:
                continue

            visited.add(current)
            group_members.append(current)

            for neighbor in adjacency_map[current]:
                if neighbor not in visited:
                    stack.append(neighbor)

        for member_id in group_members:
            group_records.append({
                csd_summary_id_col: member_id,
                "adjacency_group_id": group_id,
                "adjacency_group_size": len(group_members)
            })

        group_id += 1

    group_df = pd.DataFrame(group_records)

    # ---------------------------------------------------------------------
    # Add group-level metrics
    # ---------------------------------------------------------------------
    drop_cols = ["geometry"]

    if csd_gdf_id_col in filtered_gdf.columns and csd_gdf_id_col != csd_summary_id_col:
        drop_cols.append(csd_gdf_id_col)

    result = filtered_gdf.drop(columns=drop_cols).merge(
        group_df,
        on=csd_summary_id_col,
        how="left"
    )

    group_names = (
        result
        .groupby("adjacency_group_id")[csd_name_col]
        .apply(lambda x: ", ".join(sorted(x.astype(str).unique())))
        .reset_index()
        .rename(columns={csd_name_col: "adjacency_group_csd_names"})
    )

    group_population = (
        result
        .groupby("adjacency_group_id", as_index=False)[population_col]
        .sum()
        .rename(columns={
            population_col: "adjacency_group_population_total"
        })
    )

    def weighted_avg_distance(group: pd.DataFrame) -> float:
        valid = (
            group[travel_distance_col].notna() &
            group[population_col].notna() &
            (group[population_col] > 0)
        )

        denominator = group.loc[valid, population_col].sum()

        if denominator == 0:
            return float("nan")

        numerator = (
            group.loc[valid, population_col] *
            group.loc[valid, travel_distance_col]
        ).sum()

        return numerator / denominator

    group_weighted_distance = (
        result
        .groupby("adjacency_group_id")
        .apply(weighted_avg_distance)
        .reset_index(name="adjacency_group_weighted_avg_distance_km")
    )

    group_sehi = (
        result
        .groupby("adjacency_group_id", as_index=False)
        .agg(
            adjacency_group_min_sehi=(sehi_col, "min"),
            adjacency_group_max_sehi=(sehi_col, "max")
        )
    )

    result = result.merge(group_names, on="adjacency_group_id", how="left")
    result = result.merge(group_population, on="adjacency_group_id", how="left")
    result = result.merge(group_weighted_distance, on="adjacency_group_id", how="left")
    result = result.merge(group_sehi, on="adjacency_group_id", how="left")

    result["adjacency_group_population_total"] = (
        result["adjacency_group_population_total"].round(1)
    )

    result["adjacency_group_weighted_avg_distance_km"] = (
        result["adjacency_group_weighted_avg_distance_km"].round(2)
    )

    result["adjacency_group_min_sehi"] = (
        result["adjacency_group_min_sehi"].round(2)
    )

    result["adjacency_group_max_sehi"] = (
        result["adjacency_group_max_sehi"].round(2)
    )

    # ---------------------------------------------------------------------
    # Reorder columns
    # ---------------------------------------------------------------------
    front_cols = [
        "adjacency_group_id",
        "adjacency_group_size",
        "adjacency_group_population_total",
        "adjacency_group_weighted_avg_distance_km",
        "adjacency_group_min_sehi",
        "adjacency_group_max_sehi",
        "adjacency_group_csd_names",
        csd_summary_id_col,
        csd_name_col,
        population_col,
        sehi_col,
        travel_distance_col
    ]

    remaining_cols = [
        col for col in result.columns
        if col not in front_cols
    ]

    result = result[front_cols + remaining_cols]

    result = (
        result
        .sort_values(
            [
                "adjacency_group_id",
                "adjacency_group_population_total",
                population_col,
                travel_distance_col
            ],
            ascending=[True, False, False, False]
        )
        .reset_index(drop=True)
    )

    return result

In [ ]:
low_sehi_high_distance_groups = filter_and_group_adjacent_low_sehi_high_distance_csds(
    csd_office_summary=csd_office_summary_78,
    csd_gdf=csd_gdf,
    sehi_threshold=42,
    travel_distance_threshold_km=15,
    csd_summary_id_col="csd_id",
    csd_gdf_id_col="csd_uid"
)


# Output path
excel_path = Path(OUTPUT_DIRECTORY) / "low_sehi_high_distance_groups.xlsx"

# Save DataFrame to Excel
low_sehi_high_distance_groups.to_excel(
    excel_path,
    index=False,
    sheet_name="65 office"
)

print(f"Saved to: {excel_path}")

low_sehi_high_distance_groups.head()

In [ ]:
# Function returns percentage population under each threshold 

def summarize_population_under_thresholds(
    csd_summary: pd.DataFrame,
    total_index_threshold: float,
    weighted_avg_distance_threshold: float,
    population_col: str = "population",
    total_index_col: str = "total_index_0_100",
    distance_col: str = "weighted_avg_distance_km",
    inclusive: bool = False,
) -> pd.DataFrame:
    """
    Summarize population meeting accessibility index and distance thresholds.
 
    Parameters
    ----------
    csd_summary : pd.DataFrame
        CSD-level summary table.
    total_index_threshold : float
        Threshold applied to the total accessibility index.
    weighted_avg_distance_threshold : float
        Threshold applied to weighted average distance.
    population_col : str, default "population"
        Population column name.
    total_index_col : str, default "total_index_0_100"
        Total accessibility index column name.
    distance_col : str, default "weighted_avg_distance_km"
        Weighted average distance column name.
    inclusive : bool, default False
        If True, use <= thresholds. Otherwise, use < thresholds.
 
    Returns
    -------
    pd.DataFrame
        One-row summary table containing the requested population metrics.
    """
 
    required_columns = [
        population_col,
        total_index_col,
        distance_col,
    ]
 
    missing_columns = [
        column for column in required_columns
        if column not in csd_summary.columns
    ]
 
    if missing_columns:
        raise KeyError(
            f"csd_summary is missing required columns: {missing_columns}"
        )
 
    if total_index_threshold < 0:
        raise ValueError("total_index_threshold must be non-negative.")
 
    if weighted_avg_distance_threshold < 0:
        raise ValueError(
            "weighted_avg_distance_threshold must be non-negative."
        )
 
    working_table = csd_summary[required_columns].copy()
 
    # Change: Ensure calculation columns are numeric.
    for column in required_columns:
        working_table[column] = pd.to_numeric(
            working_table[column],
            errors="coerce",
        )
 
    # Change: Exclude rows with missing population or threshold values.
    working_table = working_table.dropna(subset=required_columns)
 
    if inclusive:
        index_mask = (
            working_table[total_index_col]
<= total_index_threshold
        )
        distance_mask = (
            working_table[distance_col]
<= weighted_avg_distance_threshold
        )
        threshold_operator = "<="
    else:
        index_mask = (
            working_table[total_index_col]
< total_index_threshold
        )
        distance_mask = (
            working_table[distance_col]
< weighted_avg_distance_threshold
        )
        threshold_operator = "<"
 
    both_mask = index_mask & distance_mask
 
    population_under_index_threshold = (
        working_table.loc[index_mask, population_col].sum()
    )
 
    population_under_both_thresholds = (
        working_table.loc[both_mask, population_col].sum()
    )
 
    if population_under_index_threshold == 0:
        population_pct_under_both_thresholds = np.nan
    else:
        population_pct_under_both_thresholds = (
            population_under_both_thresholds
            / population_under_index_threshold
            * 100
        )
 
    return pd.DataFrame(
        {
            "total_index_threshold": [total_index_threshold],
            "weighted_avg_distance_threshold": [
                weighted_avg_distance_threshold
            ],
            "threshold_operator": [threshold_operator],
            "population_under_total_index_threshold": [
                population_under_index_threshold
            ],
            "population_under_both_thresholds": [
                population_under_both_thresholds
            ],
            "population_pct_under_both_thresholds": [
                population_pct_under_both_thresholds
            ],
        }
    )

In [ ]:
# Baseline: 65 locations:
 
SEHI_summary = summarize_population_under_thresholds(
    csd_summary=csd_office_summary_78,
    total_index_threshold=42,
    weighted_avg_distance_threshold=15,
    population_col='population_total'
)
 
print(SEHI_summary)